## Test 1: Single Latent Step

In [1]:
import torch
from unsloth import FastVisionModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
WARNING 05-13 06:16:03 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from transformers import AutoProcessor

In [3]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")

In [4]:
model, tokenizer = FastVisionModel.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth",
                                                   load_in_4bit=False,
                                                   use_gradient_checkpointing="unsloth")

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.4. vLLM: 0.19.1.
   \\   /|    NVIDIA RTX A4000. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [5]:
FastVisionModel.for_inference(model)

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [6]:
prompt = "Move the red block to the left side of the table"

In [7]:
inputs = processor(text=prompt, return_tensors="pt").to(model.device)

In [8]:
with torch.no_grad():
    prompt_embeds = model.get_input_embeddings()(inputs.input_ids)

In [9]:
print(f"Prompt embeds shape: {prompt_embeds.shape}")

Prompt embeds shape: torch.Size([1, 11, 2560])


In [10]:
with torch.no_grad():
    outputs = model.model(
        inputs_embeds = prompt_embeds,
        output_hidden_states = True)
    last_hidden = outputs.last_hidden_state
    z_1 = last_hidden[:, -1:, :]

In [11]:
print(f"z_1 shape: {z_1.shape}")  # expect [1, 1, 2560]
print("Test 1 passed ✓")

z_1 shape: torch.Size([1, 1, 2560])
Test 1 passed ✓


## Test 2: Full Autoregressive Latent Loop (M=6)

In [12]:
M = 6
D = model.config.text_config.hidden_size

In [13]:
with torch.no_grad():
    current_embeds = prompt_embeds
    latents = []
    
    for m in range(M):
        outputs = model.model(
            inputs_embeds = current_embeds,
            output_hidden_states = True
        )
        z_m = outputs.last_hidden_state[:, -1:, :]
        latents.append(z_m)

        current_embeds = torch.cat([current_embeds, z_m], dim=1)

In [14]:
latents

[tensor([[[-1.5156, 14.6875, -3.2500,  ..., -1.7656, -2.4688,  1.8828]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[-4.1562,  6.5312, -0.8047,  ..., -1.6875, -0.9961,  2.4375]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[-4.0312, -1.5078, -0.1494,  ..., -0.9609,  0.5078,  2.4844]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[-3.5781, -5.4062, -0.4883,  ..., -0.5469,  1.4062,  2.6094]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[-3.0625, -6.3125, -1.0781,  ..., -0.2178,  2.0625,  2.7656]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[-2.4844, -5.9688, -1.4609,  ...,  0.0143,  2.4062,  2.6094]]],
        device='cuda:0', dtype=torch.bfloat16)]

In [15]:
z = torch.cat(latents, dim=1)  # [1, M=6, 2048]
print(f"Latent sequence shape: {z.shape}")  # expect [1, 6, 2048]
print(f"Context grew from {prompt_embeds.shape[1]} → {current_embeds.shape[1]} tokens")
print("Test 2 passed ✓")

Latent sequence shape: torch.Size([1, 6, 2560])
Context grew from 11 → 17 tokens
Test 2 passed ✓


## Test 3: Spatial Tokens in Parallel

In [16]:
K = 5

In [17]:
model.dtype

torch.bfloat16

In [18]:
spatial_embeddings = torch.nn.Embedding(K, D).to(model.device, dtype=model.dtype)
spatial_mlp = torch.nn.Linear(D, 2).to(model.device, dtype=model.dtype)

spatial_ids = torch.arange(K).unsqueeze(0).to(model.device)

In [19]:
with torch.no_grad():
    s_embeds = spatial_embeddings(spatial_ids)

    full_embeds = torch.cat([current_embeds, s_embeds], dim = 1)

    outputs = model.model(inputs_embeds = full_embeds, output_hidden_states = True)

    spatial_hidden = outputs.last_hidden_state[:, -K:, :]
    waypoints = spatial_mlp(spatial_hidden)

In [20]:
print(f"Waypoints shape: {waypoints.shape}")   # expect [1, 5, 2]
print(f"Waypoints:\n{waypoints}")
print("Test 3 passed ✓")

Waypoints shape: torch.Size([1, 5, 2])
Waypoints:
tensor([[[-1.7812,  1.1250],
         [-0.1016,  2.6719],
         [-0.4668, -1.0312],
         [-1.7422,  2.0469],
         [-1.8906, -2.5625]]], device='cuda:0', dtype=torch.bfloat16)
Test 3 passed ✓


In [21]:
full_embeds.shape

torch.Size([1, 22, 2560])

## Test 4: Gradient Flow Check

In [22]:
FastVisionModel.for_training(model)

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [23]:
spatial_embeddings = torch.nn.Embedding(K, D).to(model.device, dtype=model.dtype)
spatial_mlp = torch.nn.Linear(D, 2).to(model.device, dtype=model.dtype)

current_embeds = prompt_embeds.detach().clone()
current_embeds.requires_grad_(False)

tensor([[[-0.0212,  0.0032,  0.0332,  ..., -0.0204, -0.0116,  0.0136],
         [ 0.0177,  0.0420, -0.0603,  ...,  0.0084,  0.0064,  0.0024],
         [-0.0053,  0.0376, -0.0244,  ..., -0.0157, -0.0260, -0.0074],
         ...,
         [ 0.0199,  0.0332, -0.0486,  ...,  0.0054, -0.0069,  0.0220],
         [ 0.0177,  0.0420, -0.0603,  ...,  0.0084,  0.0064,  0.0024],
         [-0.0049,  0.0153,  0.0013,  ...,  0.0058,  0.0028,  0.0034]]],
       device='cuda:0', dtype=torch.bfloat16)

In [24]:
latents = []

In [25]:
for m in range(M):
    outputs = model.model(inputs_embeds=current_embeds, output_hidden_states=True)
    z_m = outputs.last_hidden_state[:, -1:, :]
    latents.append(z_m)
    current_embeds = torch.cat([current_embeds, z_m], dim=1)

s_embeds = spatial_embeddings(spatial_ids)
full_embeds = torch.cat([current_embeds, s_embeds], dim=1)
outputs = model.model(inputs_embeds=full_embeds, output_hidden_states=True)
spatial_hidden = outputs.last_hidden_state[:, -K:, :]
waypoints = spatial_mlp(spatial_hidden)

In [26]:
# Fake GT waypoints
gt_waypoints = torch.rand(1, K, 2).to(model.device, dtype=model.dtype)
loss = torch.nn.functional.mse_loss(waypoints, gt_waypoints)
loss.backward()

In [27]:
# Check gradients exist
print(f"Loss: {loss.item():.4f}")
print(f"spatial_mlp grad: {spatial_mlp.weight.grad is not None}")        # expect True
print(f"spatial_embed grad: {spatial_embeddings.weight.grad is not None}") # expect True
print("Test 4 passed ✓ — gradients flow through latent loop")

Loss: 3.6562
spatial_mlp grad: True
spatial_embed grad: True
Test 4 passed ✓ — gradients flow through latent loop
